<a href="https://colab.research.google.com/github/towardsai/ai-tutor-rag-system/blob/main/notebooks/17-Using_LLMs_to_rank_chunks_as_the_Judge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Install Packages and Setup Variables


In [ ]:
!pip install -q huggingface-hub llama-index==0.14.0 llama-index-llms-openai==0.5.6 llama-index-vector-stores-chroma==0.5.5 \
                llama-index-llms-google-genai==0.3.0 chromadb==1.0.12 jedi==0.19.2 ipywidgets==8.1.5

In [ ]:
import os

# Set the following API Keys in the Python environment. Will be used later.
# os.environ["OPENAI_API_KEY"] = "[OPENAI_API_KEY]"
# os.environ["GOOGLE_API_KEY"] = "<YOUR_API_KEY>"

import nest_asyncio
nest_asyncio.apply()

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
    os.environ["GOOGLE_API_KEY"] = userdata.get('Google_api_key')
except ImportError:
    os.environ.setdefault("OPENAI_API_KEY", os.environ.get("OPENAI_API_KEY", ""))
    os.environ.setdefault("GOOGLE_API_KEY", os.environ.get("GOOGLE_API_KEY", ""))

# Load a Model


In [ ]:
from llama_index.llms.openai import OpenAI
from llama_index.core import Settings
from llama_index.embeddings.openai import OpenAIEmbedding

Settings.llm = OpenAI(model="gpt-4o-mini")
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

# Load Indexes


Downloading vector store from Huggingface hub

In [ ]:
import shutil
from huggingface_hub import hf_hub_download

_local_vs = "/Users/jai/Documents/code-repo/ai-engineering-v2-notebooks-claude-skill/ai-engineering-v2-notebooks-claude-skill/notebooks/vectorstore.zip"
if os.path.exists(_local_vs):
    shutil.copy(_local_vs, "./vectorstore.zip")
    vectorstore = "./vectorstore.zip"
else:
    vectorstore = hf_hub_download(repo_id="jaiganesan/ai_tutor_knowledge", filename="vectorstore.zip", repo_type="dataset", local_dir=".")

import zipfile
if not os.path.exists("./ai_tutor_knowledge"):
    with zipfile.ZipFile("./vectorstore.zip", 'r') as zf:
        zf.extractall(".")

In [ ]:
# Load the vector store from the local storage.
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore

db2 = chromadb.PersistentClient(path="./ai_tutor_knowledge")
chroma_collection = db2.get_or_create_collection("ai_tutor_knowledge")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

In [ ]:
from llama_index.core import VectorStoreIndex
from llama_index.embeddings.openai import OpenAIEmbedding

# Create the index based on the vector store.
index = VectorStoreIndex.from_vector_store(vector_store=vector_store)

In [ ]:
query_engine = index.as_query_engine(similarity_top_k=10)

res = query_engine.query("Explain how Advance RAG works?")

In [ ]:
res.response

# RankGPT


In [ ]:
from llama_index.core.postprocessor.rankGPT_rerank import RankGPTRerank

rankGPT = RankGPTRerank(top_n=3,llm=Settings.llm, verbose=True)

In [ ]:
# Define a query engine that is responsible for retrieving related pieces of text,
# and using a LLM to formulate the final answer.
# The `node_postprocessors` function will be applied to the retrieved nodes.
query_engine = index.as_query_engine(similarity_top_k=10, node_postprocessors=[rankGPT])

res = query_engine.query("Explain how Retrieval Augmented Generation (RAG) works?")

In [ ]:
res.response

In [ ]:
# Show the retrieved nodes
for src in res.source_nodes:
    print("Node ID\t", src.node_id)
    print("Title\t", src.metadata["title"])
    print("Text\t", src.text.strip())
    print("Score\t", src.score)
    print("-_" * 20)

# Custom Postprocessor


## The `Judger` Function


The following function will query GPT-4o-mini to retrieve the top three nodes that have the highest similarity to the asked question.


In [ ]:
from pydantic import BaseModel
from llama_index.llms.openai import OpenAI
from llama_index.core.prompts import PromptTemplate


def judger(nodes, query):

    # The model's output template
    class OrderedNodes(BaseModel):
        """A node with the id and assigned score."""

        node_id: list
        score: list

    # Prepare the nodes and wrap them in <NODE></NODE> identifier, as well as the query
    the_nodes = ""
    for idx, item in enumerate(nodes):
        the_nodes += f"<NODE{idx+1}>\nNode ID: {item.node_id}\nText: {item.text}\n</NODE{idx+1}>\n"

    query = "<QUERY>\n{}\n</QUERY>".format(query)

    # Define the prompt template
    prompt_tmpl = PromptTemplate(
        """
    You receive a query along with a list of nodes' text and their ids. Your task is to assign a score
    to each node based on its contextual closeness to the given query. The final output is each
    node id along with its proximity score.
    Here is the list of nodes:
    {nodes_list}

    And the following is the query:
    {user_query}

    Score each of the nodes based on their text and their relevancy to the provided query.
    The score must be a decimal number between 0 and 1 so we can rank them."""
    )

    # Define the an instance of GPT-5 mini and send the request
    llm = OpenAI(model="gpt-4o-mini")
    ordered_nodes = llm.structured_predict(
        OrderedNodes, prompt_tmpl, nodes_list=the_nodes, user_query=query
    )

    return ordered_nodes

## Define Postprocessor


The following class will use the `judger` function to rank the nodes, and filter them based on the ranks.


In [ ]:
from typing import List, Optional
from llama_index.core import QueryBundle
from llama_index.core.postprocessor.types import BaseNodePostprocessor
from llama_index.core.schema import NodeWithScore


class OpenaiAsJudgePostprocessor(BaseNodePostprocessor):
    def _postprocess_nodes(
        self, nodes: List[NodeWithScore], query_bundle: Optional[QueryBundle]
    ) -> List[NodeWithScore]:

        query_str = query_bundle.query_str if query_bundle else ""
        r = judger(nodes, query_str)

        node_ids = r.node_id
        scores = r.score

        sorted_scores = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
        num_nodes_to_select = min(3, len(sorted_scores))
        top_nodes = [sorted_scores[i][0] for i in range(num_nodes_to_select)]
        # top_nodes = [sorted_scores[i][0] for i in range(3)]

        selected_nodes_id = [node_ids[item] for item in top_nodes]

        final_nodes = []
        for item in nodes:
            if item.node_id in selected_nodes_id:
                final_nodes.append(item)

        return final_nodes

In [ ]:
judge = OpenaiAsJudgePostprocessor()

## Query Engine with Postprocessor


In [ ]:
# Define a query engine that is responsible for retrieving related pieces of text,
# and using a LLM to formulate the final answer.
# The `node_postprocessors` function will be applied to the retrieved nodes.
query_engine = index.as_query_engine(similarity_top_k=10, node_postprocessors=[judge])

res = query_engine.query("Compare Retrieval Augmented Generation (RAG) and Parameter efficient Finetuning (PEFT)")

In [ ]:
res.response

In [ ]:
# Show the retrieved nodes
for src in res.source_nodes:
    print("Node ID\t", src.node_id)
    print("Title\t", src.metadata["title"])
    print("Text\t", src.text)
    print("Score\t", src.score)
    print("-_" * 20)

## Optional: Interactive LLM-as-Judge Ranking Demo

Use the widget below to submit a custom query and compare raw retrieval vs. LLM-judged re-ranked results.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display

    query_input = widgets.Text(
        value="Explain how RAG works",
        description="Query:",
        layout=widgets.Layout(width="70%"),
        style={"description_width": "initial"},
    )
    run_btn = widgets.Button(description="Run & Compare", button_style="primary")
    out = widgets.Output(layout=widgets.Layout(border="1px solid #ccc", padding="8px"))

    def on_click(b):
        out.clear_output()
        with out:
            q = query_input.value.strip()
            print(f"Query: {q}\n")

            # Raw retrieval (no reranking)
            raw_engine = index.as_query_engine(similarity_top_k=5)
            raw_res = raw_engine.query(q)
            print("--- Raw Retrieval (top 5) ---")
            for i, src in enumerate(raw_res.source_nodes, 1):
                print(f"  {i}. {src.metadata.get('title', src.node_id)} (score={src.score:.4f})")

            # LLM-judged re-ranking
            judged_engine = index.as_query_engine(similarity_top_k=5, node_postprocessors=[judge])
            judged_res = judged_engine.query(q)
            print("\n--- LLM-Judged Re-ranking (top 3) ---")
            for i, src in enumerate(judged_res.source_nodes, 1):
                print(f"  {i}. {src.metadata.get('title', src.node_id)} (score={src.score:.4f})")

            print(f"\nFinal Answer:\n{judged_res.response}")

    run_btn.on_click(on_click)
    display(widgets.VBox([query_input, run_btn, out]))
except ImportError:
    print("ipywidgets not installed. Run: pip install ipywidgets")